# PVG matched control (no provers)

Trains the same verifier as the v5 game, with the same warm-up, calibration, optimizer and seeding, for 7 rounds. Each round uses exactly as many training examples as the game's verifier used in that round (fixed dataset plus prover write-ups, 434 to 928), but every example comes from the fixed labeled dataset. No provers are trained or sampled.

It reports held-out accuracy and AUROC after the warm-up (round 0) and after every round, on heldout_100, heldout_48 and spot_20, next to the game's own heldout_100 numbers from the v5 logs. If the game's round-7 verifier beats this control, the gain comes from what the provers wrote.

Runtime: A100. About 15 minutes of setup, then roughly 10 to 15 minutes per seed (5 seeds). Results are saved after each seed to `eval/matched_control_v5.json`, copied to `MyDrive/pvg_v5_results` and downloaded. Re-running the last cell after a disconnect skips the seeds already done, as long as the runtime still has that file.

## 1. Setup

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/gunasti2002/Automated_R_leg_NoAPI.git"
BRANCH    = "main"
CLONE_DIR = Path("/content/Automated_R_leg_NoAPI")

%cd /content
!rm -rf {CLONE_DIR}
!git clone --branch {BRANCH} {REPO_URL} {CLONE_DIR}

matches = sorted(CLONE_DIR.rglob("config.py"))
assert matches, f"No config.py found under {CLONE_DIR}"
PROJECT_ROOT = matches[0].parent
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
!git -C {CLONE_DIR} log --oneline -3


In [ ]:
# Colab ships torchao 0.10, which peft refuses (needs >= 0.16). Nothing here uses it: remove it.
!pip uninstall -y -q torchao
# torch + matplotlib ship with Colab; do not reinstall torch (CUDA wheel mismatch risk).
!pip install -q jinja2 "transformers>=4.50" peft accelerate
import importlib; importlib.invalidate_caches()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

# bitsandbytes for the 4-bit 32B judge
!pip -q install -U bitsandbytes

### Brandon's fixes

Pins the project code to 51940d1, patches it, rebuilds the dataset with the 10 real records and writes the Qwen-written rows (GPU, about 15 minutes). In the output, check that every patch line starts with `+` and that both model-written files have rows. The data cell below rebuilds the same dataset, which is harmless.

In [ ]:
!python apply_fixes.py
!git -C {CLONE_DIR} status --short | head -20

In [ ]:
# Brandon's change 3: a stronger learning signal for the prover. Subprocesses inherit this.
import os, json
OVERRIDES = {
    "prover_lr": 1e-4,                 # was 2e-5
    "prover_num_generations": 8,       # was 4 samples per record
    "findings_per_round": 16,          # was 8 records per round
    "prover_kl_penalty": 0.05,         # was 0.02
    # Both roles gated by the rule check: helpful earns P(sound) only if it passes,
    # sneaky only if it fails. Matches the verifier labels (sneaky write-ups that pass
    # the rule check are dropped from verifier training, so they should earn nothing).
    "prover_reward_mode": "correctness_gated",
    # v5: 3 warm-start epochs (apply_fixes default is 2; v4 reached helpful fidelity 0.27-0.41)
    "prover_sft_epochs": 3,
}
os.environ["PVG_CONFIG_OVERRIDES"] = json.dumps(OVERRIDES)

import sys; sys.path.insert(0, ".")
from config import PVGConfig
cfg = PVGConfig()
print({k: getattr(cfg, k) for k in OVERRIDES})
print("samples per round:", 2 * cfg.findings_per_round * cfg.prover_num_generations)


## 2. Data

120 honest/sneaky pairs, 20% held out by record, seed 42 (unchanged).

In [ ]:
!python data/build_findings_dataset.py --target-pairs 120 --seed 42 \
    --out data/findings_pvg_dataset.jsonl --heldout-out data/findings_pvg_heldout.jsonl \
    --manifest-out data/dataset_manifest.json | tail -4


## Matched control

In [ ]:
# Matched control: the same verifier, warm-up, calibration and optimizer as the v5 game, trained each
# round on exactly as many examples as the game used, but drawn only from the fixed labeled dataset
# (templated rows + Qwen-written rows). No provers. Any gain of the game's round-7 verifier over this
# control comes from what the provers wrote.
import gc, json, random, shutil, time
from pathlib import Path

import torch
from config import PVGConfig
from training import pvg_loop

SEEDS = [42, 43, 44, 45, 46]
ROUNDS = 7
# Examples the game's verifier trained on in each round (fixed dataset + prover replay), from the v5 logs.
GAME_EXAMPLES = {42: [454, 566, 716, 872, 928, 928, 928], 43: [462, 570, 698, 846, 928, 928, 928], 44: [434, 554, 710, 870, 928, 928, 928], 45: [416, 544, 698, 866, 928, 928, 928], 46: [424, 530, 638, 780, 928, 928, 928]}
# The game's held-out accuracy / AUROC on heldout_100 at each round, from the v5 logs.
GAME_HELDOUT = {42: [(0.91, 0.975), (0.95, 0.988), (0.94, 0.977), (0.97, 0.98), (0.91, 0.984), (0.96, 0.987), (0.92, 0.999)],
                43: [(0.91, 0.974), (0.91, 0.982), (0.94, 0.983), (0.94, 0.99), (0.95, 0.995), (0.96, 0.99), (0.93, 0.985)],
                44: [(0.9, 0.975), (0.95, 0.98), (0.94, 0.982), (0.93, 0.991), (0.9, 0.986), (0.97, 0.978), (0.95, 0.977)],
                45: [(0.93, 0.98), (0.95, 0.984), (0.98, 0.998), (0.96, 0.982), (0.98, 0.986), (0.96, 0.989), (0.98, 0.986)],
                46: [(0.92, 0.974), (0.88, 0.958), (0.96, 0.989), (0.95, 0.996), (0.98, 0.995), (0.94, 0.993), (0.98, 0.996)]}
OUT = Path("eval/matched_control_v5.json")
DRIVE_DIR = "/content/drive/MyDrive/pvg_v5_results"

cfg = PVGConfig()
load = pvg_loop.load_dataset
dataset = load(cfg.output_dataset_path)
verifier_rows = pvg_loop.with_model_written(dataset, cfg.model_written_train_path, "train")
h48 = load(cfg.heldout_dataset_path)
SETS = {"heldout_100": pvg_loop.with_model_written(h48, cfg.model_written_heldout_path, "held-out"),
        "heldout_48": h48, "spot_20": load(cfg.spot_check_set_path)}
spot_rows = SETS["spot_20"]
print({k: len(v) for k, v in SETS.items()}, "| verifier pool:", len(verifier_rows))


def evaluate():
    out = {}
    for name, rows in SETS.items():
        ev = pvg_loop.evaluate_verifier_on_rows(rows, cfg)
        preds = [s >= pvg_loop.ACCEPT_THRESHOLD for s in ev["scores"]]
        n_u = sum(1 for l in ev["labels"] if not l)
        out[name] = {"accuracy": ev["accuracy"], "auroc": ev["auroc"], "accept_rate": ev["accept_rate"],
                     "sneaky_reject_rate": sum(1 for p, l in zip(preds, ev["labels"]) if not l and not p) / max(1, n_u)}
    return out


results = json.loads(OUT.read_text()) if OUT.exists() else {}
for _s in results.values():   # JSON turns round keys into strings
    _s["rounds"] = {int(k): v for k, v in _s["rounds"].items()}
for seed in SEEDS:
    if str(seed) in results and len(results[str(seed)]["rounds"]) == ROUNDS + 1:
        print(f"seed {seed}: done already, skipping"); continue
    t0 = time.time()
    cfg.seed = seed
    pvg_loop._VERIFIER_CACHE.clear(); pvg_loop._DATASET_EXAMPLES.clear(); pvg_loop._REPLAY_BUFFER.clear()
    gc.collect(); torch.cuda.empty_cache()
    random.seed(seed * 1000 + 1); torch.manual_seed(seed * 1000 + 1)   # same seeding as the game's round 1
    print(f"\n===== seed {seed}: warm-up (identical to the game) =====")
    pvg_loop.prepare_verifier_for_run(cfg, verifier_rows, spot_rows)
    rounds = {0: {"examples": 0, **evaluate()}}
    print(f"  round 0 (after warm-up): heldout_100 acc {rounds[0]['heldout_100']['accuracy']:.3f}  "
          f"heldout_48 acc {rounds[0]['heldout_48']['accuracy']:.3f}  spot_20 acc {rounds[0]['spot_20']['accuracy']:.3f}")
    base = list(pvg_loop._DATASET_EXAMPLES)
    for r in range(1, ROUNDS + 1):
        random.seed(seed * 1000 + r); torch.manual_seed(seed * 1000 + r)
        n = GAME_EXAMPLES[seed][r - 1]
        rng = random.Random(seed * 1000 + r)
        # every fixed example once, topped up with random draws from the same fixed pool to match the game's count exactly
        examples = (base + [rng.choice(base) for _ in range(n - len(base))]) if n >= len(base) else rng.sample(base, n)
        assert len(examples) == n
        stats = pvg_loop.train_verifier_on_examples(examples, cfg, epochs=cfg.verifier_train_epochs_per_round, tag=f"control r{r}")
        rounds[r] = {"examples": len(examples), "loss": stats.get("loss"), "logit_bias": stats.get("logit_bias"), **evaluate()}
        g = GAME_HELDOUT[seed][r - 1]
        print(f"  round {r}: {len(examples)} examples | control heldout_100 acc {rounds[r]['heldout_100']['accuracy']:.3f} "
              f"AUROC {rounds[r]['heldout_100']['auroc']:.3f} | game {g[0]:.3f} / {g[1]:.3f} | "
              f"heldout_48 {rounds[r]['heldout_48']['accuracy']:.3f} | spot_20 {rounds[r]['spot_20']['accuracy']:.3f}")
    results[str(seed)] = {"rounds": rounds, "seconds": time.time() - t0}
    OUT.parent.mkdir(parents=True, exist_ok=True)
    OUT.write_text(json.dumps(results, indent=1))      # saved after every seed, so a disconnect loses at most one seed
    print(f"  seed {seed} done in {(time.time() - t0) / 60:.1f} min, saved to {OUT}")

# ---- summary: mean over seeds ---------------------------------------------------------------------
import statistics as st
seeds = [s for s in SEEDS if str(s) in results]
print(f"\n### Matched control vs the game, mean over {len(seeds)} seeds")
print("| round | control heldout_100 acc | control AUROC | game heldout_100 acc | game AUROC | control heldout_48 acc | control spot_20 acc |")
print("|---|---|---|---|---|---|---|")
for r in range(0, ROUNDS + 1):
    c = [results[str(s)]["rounds"][r] for s in seeds]
    m = lambda k, sub: st.mean(x[k][sub] for x in c)
    game = (f"{st.mean(GAME_HELDOUT[s][r - 1][0] for s in seeds):.3f} | {st.mean(GAME_HELDOUT[s][r - 1][1] for s in seeds):.3f}"
            if r else "- | -")
    print(f"| {r} | {m('heldout_100', 'accuracy'):.3f} | {m('heldout_100', 'auroc'):.3f} | {game} | "
          f"{m('heldout_48', 'accuracy'):.3f} | {m('spot_20', 'accuracy'):.3f} |")
print("\nThe game's own numbers are from the v5 logs (heldout_100 only). Its Qwen-written held-out rows were sampled in that run,"
      " so they can differ by a few rows from the ones regenerated here.")

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    shutil.copy(OUT, f"{DRIVE_DIR}/{OUT.name}")
    print("Copied to", f"{DRIVE_DIR}/{OUT.name}")
except Exception as e:
    print("Drive copy skipped:", e)
try:
    from google.colab import files
    files.download(str(OUT))
except Exception as e:
    print("Download skipped:", e)
